In [4]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

BASE_DIR = Path().resolve().parent

seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

encoding_dim_list = [4, 8]
hidden_dim_list = [16, 32]
batch_size_list = [32, 64]
learning_rate_list = [1e-3, 5e-4]
threshold_quantile_list = [0.10, 0.15, 0.20, 0.25, 0.30]

epochs = 100
patience = 10
window_size = 50

data_path = BASE_DIR / "data_processed/NAB/industrial_machine_features_labeled.csv"
results_root = BASE_DIR / "results/AE_NAB"
results_root.mkdir(parents=True, exist_ok=True)

feature_cols = ["value", "diff1", "roll_mean", "roll_std", "roll_min", "roll_max"]

df = pd.read_csv(data_path)
required_cols = feature_cols + ["y_true"]
df = df.dropna(subset=required_cols).reset_index(drop=True)

if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

X = df[feature_cols].copy()
y_true = df["y_true"].astype(int).values


def create_windows(X_array, y_array, window_size):
    X_windows = []
    y_windows = []
    end_indices = []
    for i in range(len(X_array) - window_size + 1):
        X_windows.append(X_array[i:i + window_size])
        y_windows.append(1 if np.any(y_array[i:i + window_size] == 1) else 0)
        end_indices.append(i + window_size - 1)
    return np.array(X_windows), np.array(y_windows), np.array(end_indices)


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_windows, y_windows, end_indices = create_windows(
    X_scaled,
    y_true,
    window_size
)

X_windows_flat = X_windows.reshape(X_windows.shape[0], -1)

X_train = X_windows_flat[y_windows == 0]
X_test = X_windows_flat

input_dim = X_train.shape[1]


def to_str(value):
    if isinstance(value, str):
        return value
    return str(value).replace(".", "_")


def build_autoencoder(input_dim, hidden_dim, encoding_dim, learning_rate):
    inputs = Input(shape=(input_dim,))
    x = Dense(hidden_dim, activation="relu")(inputs)
    latent = Dense(encoding_dim, activation="relu")(x)
    x = Dense(hidden_dim, activation="relu")(latent)
    outputs = Dense(input_dim, activation="linear")(x)
    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse"
    )
    return model


all_results = []

for hidden_dim in hidden_dim_list:
    hidden_dir = results_root / f"hidden_{to_str(hidden_dim)}"
    hidden_dir.mkdir(parents=True, exist_ok=True)

    hidden_results = []

    for encoding_dim in encoding_dim_list:
        encoding_dir = hidden_dir / f"latent_{to_str(encoding_dim)}"
        encoding_dir.mkdir(parents=True, exist_ok=True)

        encoding_results = []

        for batch_size in batch_size_list:
            batch_dir = encoding_dir / f"batch_{to_str(batch_size)}"
            batch_dir.mkdir(parents=True, exist_ok=True)

            batch_results = []

            for learning_rate in learning_rate_list:
                lr_dir = batch_dir / f"lr_{to_str(learning_rate)}"
                lr_dir.mkdir(parents=True, exist_ok=True)

                tf.keras.backend.clear_session()

                model = build_autoencoder(
                    input_dim=input_dim,
                    hidden_dim=hidden_dim,
                    encoding_dim=encoding_dim,
                    learning_rate=learning_rate
                )

                early_stopping = EarlyStopping(
                    monitor="val_loss",
                    patience=patience,
                    restore_best_weights=True
                )

                history = model.fit(
                    X_train,
                    X_train,
                    epochs=epochs,
                    batch_size=batch_size,
                    validation_split=0.2,
                    shuffle=True,
                    callbacks=[early_stopping],
                    verbose=0
                )

                train_recon = model.predict(X_train, verbose=0)
                train_errors = np.mean(np.square(X_train - train_recon), axis=1)

                test_recon = model.predict(X_test, verbose=0)
                test_errors = np.mean(np.square(X_test - test_recon), axis=1)

                for threshold_quantile in threshold_quantile_list:
                    exp_name = (
                        f"window_{window_size}"
                        f"_latent_{to_str(encoding_dim)}"
                        f"_hidden_{to_str(hidden_dim)}"
                        f"_batch_{to_str(batch_size)}"
                        f"_lr_{to_str(learning_rate)}"
                        f"_q_{to_str(threshold_quantile)}"
                    )

                    exp_dir = lr_dir / f"q_{to_str(threshold_quantile)}"
                    exp_dir.mkdir(parents=True, exist_ok=True)

                    threshold = float(np.quantile(train_errors, threshold_quantile))
                    y_pred_windows = (test_errors > threshold).astype(int)

                    cm = confusion_matrix(y_windows, y_pred_windows)

                    accuracy = accuracy_score(y_windows, y_pred_windows) * 100
                    precision = precision_score(y_windows, y_pred_windows, zero_division=0) * 100
                    recall = recall_score(y_windows, y_pred_windows, zero_division=0) * 100
                    f1 = f1_score(y_windows, y_pred_windows, zero_division=0) * 100

                    epochs_trained = len(history.history["loss"])
                    best_val_loss = float(np.min(history.history["val_loss"]))
                    final_train_loss = float(history.history["loss"][-1])

                    result_row = {
                        "experiment": exp_name,
                        "window_size": window_size,
                        "hidden_dim": hidden_dim,
                        "encoding_dim": encoding_dim,
                        "batch_size": batch_size,
                        "learning_rate": learning_rate,
                        "threshold_quantile": threshold_quantile,
                        "threshold": threshold,
                        "epochs_trained": epochs_trained,
                        "best_val_loss": round(best_val_loss, 8),
                        "final_train_loss": round(final_train_loss, 8),
                        "accuracy_%": round(accuracy, 2),
                        "precision_%": round(precision, 2),
                        "recall_%": round(recall, 2),
                        "f1_%": round(f1, 2)
                    }

                    all_results.append(result_row)
                    hidden_results.append(result_row)
                    encoding_results.append(result_row)
                    batch_results.append(result_row)

                    df_windows = df.iloc[end_indices].copy().reset_index(drop=True)
                    df_windows["window_end_index"] = end_indices
                    df_windows["y_window_true"] = y_windows
                    df_windows["reconstruction_error"] = test_errors
                    df_windows["threshold"] = threshold
                    df_windows["is_anomaly"] = y_pred_windows

                    df_windows.to_csv(exp_dir / "predictions.csv", index=False)

                    history_df = pd.DataFrame(history.history)
                    history_df.to_csv(exp_dir / "history.csv", index=False)

                    plt.figure(figsize=(10, 4))
                    plt.plot(np.arange(len(test_errors)), test_errors)
                    plt.axhline(y=threshold)
                    plt.savefig(exp_dir / "reconstruction_error.png", dpi=300, bbox_inches="tight")
                    plt.close()

                pd.DataFrame(batch_results).sort_values(by="f1_%", ascending=False).reset_index(drop=True).to_csv(lr_dir / "summary_results.csv", index=False)

        pd.DataFrame(encoding_results).sort_values(by="f1_%", ascending=False).reset_index(drop=True).to_csv(encoding_dir / "summary_results.csv", index=False)

    pd.DataFrame(hidden_results).sort_values(by="f1_%", ascending=False).reset_index(drop=True).to_csv(hidden_dir / "summary_results.csv", index=False)

pd.DataFrame(all_results).sort_values(by="f1_%", ascending=False).reset_index(drop=True).to_csv(results_root / "summary_results.csv", index=False)

print("Done")

Done
